Prueba

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [2]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\OneDrive\\Escritorio\\computadorNuevo\\SNpollutionFinal.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [3]:
datos.head()

,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-01 00:00:00,129.0,-21,-11.0,1021.0,2,1.79
2010-01-01 01:00:00,129.0,-21,-12.0,1020.0,2,4.92
2010-01-01 02:00:00,129.0,-21,-11.0,1019.0,2,6.71
2010-01-01 03:00:00,129.0,-21,-14.0,1019.0,2,9.84
2010-01-01 04:00:00,129.0,-20,-12.0,1018.0,2,12.97


Se eliminan las primeras 24 filas debido a que estas contenían valores NAN en la columna pollution, y habían sido rellenadas con interpolación lineal.

In [4]:
datos = datos.drop(datos.index[:24])

datos

,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-02 00:00:00,129.0,-16,-4.0,1020.0,1,1.79
2010-01-02 01:00:00,148.0,-15,-4.0,1020.0,1,2.68
2010-01-02 02:00:00,159.0,-11,-5.0,1021.0,1,3.57
2010-01-02 03:00:00,181.0,-7,-5.0,1022.0,1,5.36
2010-01-02 04:00:00,138.0,-7,-5.0,1022.0,1,6.25
...,...,...,...,...,...,...
2014-12-31 19:00:00,8.0,-23,-2.0,1034.0,2,231.97
2014-12-31 20:00:00,10.0,-22,-3.0,1034.0,2,237.78
2014-12-31 21:00:00,10.0,-22,-3.0,1034.0,2,242.70


In [5]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Se divide el dataset

In [6]:
# Dividir el conjunto de datos en entrenamiento y prueba
train, test = train_test_split(datos, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
test, val = train_test_split(test, test_size=0.33, shuffle=False)

print("Las dimensiones de train son: ", train.shape)
print("Las dimensiones de test son: ", test.shape)
print("Las dimensiones de val son: ", val.shape)

Las dimensiones de train son:  (30660, 6)
Las dimensiones de test son:  (8803, 6)
Las dimensiones de val son:  (4337, 6)


Se normalizan los datos

In [7]:
from sklearn.preprocessing import StandardScaler


# Normalizar solo con los datos de entrenamiento
scaler = StandardScaler()
train = scaler.fit_transform(train)

# Aplicar la transformación a test y val usando los parámetros de train
test = scaler.transform(test)
val = scaler.transform(val)

train = pd.DataFrame(train, columns=datos.columns)
test = pd.DataFrame(test, columns=datos.columns)
val = pd.DataFrame(val, columns=datos.columns)


Se unen los datos nuevamente, ahora normalizados, en un único conjunto.

In [8]:
datosNormalizados = pd.concat([train, test, val])

datosNormalizados.index = datos.index


In [9]:
datosNormalizados.shape

(43800, 6)

In [10]:
datosNormalizados.head(10)


,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-02 00:00:00,0.317681,-1.214023,-1.268524,0.329687,-0.380944,-0.464048
2010-01-02 01:00:00,0.526152,-1.144302,-1.268524,0.329687,-0.380944,-0.446575
2010-01-02 02:00:00,0.646846,-0.865419,-1.349314,0.426127,-0.380944,-0.429103
2010-01-02 03:00:00,0.888234,-0.586536,-1.349314,0.522567,-0.380944,-0.393962
2010-01-02 04:00:00,0.416431,-0.586536,-1.349314,0.522567,-0.380944,-0.376489
2010-01-02 05:00:00,0.098238,-0.586536,-1.430104,0.522567,-0.380944,-0.359017
2010-01-02 06:00:00,0.054349,-0.586536,-1.430104,0.619008,-0.380944,-0.323876
2010-01-02 07:00:00,0.262820,-0.586536,-1.349314,0.715448,-0.380944,-0.288735
2010-01-02 08:00:00,0.218931,-0.656257,-1.430104,0.715448,-0.380944,-0.253594


Espacio de búsqueda

In [11]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas GRU
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades GRU
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes GRU, es decir [observaciones, retardos, caracteristicas]

In [12]:
futuros = 1
pasados  = 12

In [13]:
datosX = []
datosY = []
for i in range(pasados, len(datosNormalizados) - futuros + 1):
  datosX.append(datosNormalizados.iloc[i-pasados:i, 0:datosNormalizados.shape[1]])
  datosY.append(datosNormalizados.iloc[i+futuros-1:i+futuros, 0])


In [14]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (43788, 12, 6)
Dimensiones de Y: (43788, 1)


In [15]:
inputs = datosX.shape[1] * datosX.shape[2]
datosX = datosX.reshape(datosX.shape[0], inputs)

In [16]:
print("Dimensiones de X después de rehape:", datosX.shape)

Dimensiones de X después de rehape: (43788, 72)


Se dividen nuevamente los conjuntos de datos

In [17]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (30651, 72)
Las dimensiones de testX son:  (8801, 72)
Las dimensiones de valX son:  (4336, 72)


In [18]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (30651, 1)
Las dimensiones de testY son:  (8801, 1)
Las dimensiones de valY son:  (4336, 1)


Se crean métricas para medir desempeño

In [19]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

In [20]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae

def graficarPrediccion(modelo, x, y, inicio, final):
  predicciones = modelo.predict(x)
  predicciones = predicciones.flatten()
  df = pd.DataFrame({'Originales': y, 'Predichos': predicciones})
  plt.plot(df.index, df['Originales'][inicio:final], label='Originales')
  plt.plot(df.index, df['Predichos'][inicio:final], label='Predichos')
  return df, mse(y, predicciones),  mae(y, predicciones), rmse(y, predicciones), smape(y,predicciones), ia(y, predicciones)

Versión Final


In [21]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(trainX.shape[1],)))
    if (params['layers'] == 1):
      model.add(Dense(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(Dense(units=params['units'], activation=params['activation']))
          model.add(Dropout(params['dropout']))
      model.add(Dense(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(trainX, trainY, epochs=128,
                        validation_data=(testX, testY),
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [22]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=12, trials=trials, rstate=np.random.default_rng(42))

print(best)

  0%|          | 0/12 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

240/240 - 8s - 34ms/step - ia: 0.2514 - loss: 1.7808 - mae: 1.1324 - rmse: 1.3275 - smape: 1.5555 - val_ia: 0.2529 - val_loss: 1.2191 - val_mae: 0.9412 - val_rmse: 1.0625 - val_smape: 1.5971

Epoch 2/128                                           

240/240 - 1s - 5ms/step - ia: 0.2356 - loss: 1.2554 - mae: 0.8943 - rmse: 1.1156 - smape: 1.5463 - val_ia: 0.2596 - val_loss: 0.9917 - val_mae: 0.7842 - val_rmse: 0.9141 - val_smape: 1.7066

Epoch 3/128                                           

240/240 - 1s - 3ms/step - ia: 0.2373 - loss: 1.1468 - mae: 0.8220 - rmse: 1.0659 - smape: 1.5157 - val_ia: 0.2582 - val_loss: 0.9502 - val_mae: 0.7394 - val_rmse: 0.8740 - val_smape: 1.8387

Epoch 4/128                                           

240/240 - 1s - 3ms/step - ia: 0.2394 - loss: 1.1118 - mae: 0.7967 - rmse: 1.0492 - smape: 1.5070 - val_ia: 0.2631 - val_loss: 0.9311 - val_mae: 0.7243 - val_rmse: 0.8596 - val_smape: 1.8547

Epoch 5/128

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                      

1916/1916 - 10s - 5ms/step - ia: 0.8497 - loss: 0.1011 - mae: 0.2026 - rmse: 0.2834 - smape: 0.4712 - val_ia: 0.6375 - val_loss: 0.0642 - val_mae: 0.1557 - val_rmse: 0.2101 - val_smape: 0.4016

Epoch 2/128                                                                      

1916/1916 - 11s - 6ms/step - ia: 0.8728 - loss: 0.0772 - mae: 0.1730 - rmse: 0.2491 - smape: 0.4215 - val_ia: 0.6520 - val_loss: 0.0632 - val_mae: 0.1528 - val_rmse: 0.2058 - val_smape: 0.3905

Epoch 3/128                                                                      

1916/1916 - 7s - 4ms/step - ia: 0.8765 - loss: 0.0751 - mae: 0.1685 - rmse: 0.2449 - smape: 0.4096 - val_ia: 0.6509 - val_loss: 0.0624 - val_mae: 0.1487 - val_rmse: 0.2045 - val_smape: 0.3906

Epoch 4/128                                                                      

1916/1916 - 11s - 6ms/step - ia: 0.8788 - loss: 0.0732 - mae: 0.1639 - rmse: 0.2408 - s

...

Epoch 1/128                                                                         

1916/1916 - 10s - 5ms/step - ia: 0.5398 - loss: 0.6912 - mae: 0.6159 - rmse: 0.7947 - smape: 1.0663 - val_ia: 0.4172 - val_loss: 0.2253 - val_mae: 0.3011 - val_rmse: 0.3653 - val_smape: 0.6366

Epoch 2/128                                                                         

1916/1916 - 6s - 3ms/step - ia: 0.6921 - loss: 0.3320 - mae: 0.4134 - rmse: 0.5487 - smape: 0.8055 - val_ia: 0.4804 - val_loss: 0.1522 - val_mae: 0.2427 - val_rmse: 0.3038 - val_smape: 0.5541

Epoch 3/128                                                                         

1916/1916 - 5s - 3ms/step - ia: 0.7465 - loss: 0.2310 - mae: 0.3347 - rmse: 0.4547 - smape: 0.7024 - val_ia: 0.4974 - val_loss: 0.1205 - val_mae: 0.2256 - val_rmse: 0.2831 - val_smape: 0.5324

Epoch 4/128                                                                         

1916/1916 - 5s - 2ms/step - ia: 0.7705 - loss: 0.1895 - mae: 0.3007 - rmse: 

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                          

3832/3832 - 13s - 3ms/step - ia: 0.2334 - loss: 1.9477 - mae: 1.0337 - rmse: 1.3055 - smape: 1.6663 - val_ia: 0.1719 - val_loss: 1.0912 - val_mae: 0.8375 - val_rmse: 0.8843 - val_smape: 1.6368

Epoch 2/128                                                                          

3832/3832 - 9s - 2ms/step - ia: 0.2365 - loss: 1.8198 - mae: 1.0082 - rmse: 1.2666 - smape: 1.6617 - val_ia: 0.1740 - val_loss: 1.0521 - val_mae: 0.8196 - val_rmse: 0.8659 - val_smape: 1.6366

Epoch 3/128                                                                          

3832/3832 - 9s - 2ms/step - ia: 0.2375 - loss: 1.7562 - mae: 0.9895 - rmse: 1.2416 - smape: 1.6625 - val_ia: 0.1759 - val_loss: 1.0191 - val_mae: 0.8040 - val_rmse: 0.8499 - val_smape: 1.6366

Epoch 4/128                                                                          

3832/3832 - 10s - 3ms/step - ia: 0.2403 - loss: 1.6526 - mae: 0.9648 - r

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                          

240/240 - 5s - 21ms/step - ia: 0.6157 - loss: 0.4985 - mae: 0.5160 - rmse: 0.6931 - smape: 0.9676 - val_ia: 0.7230 - val_loss: 0.1942 - val_mae: 0.2862 - val_rmse: 0.3991 - val_smape: 0.6385

Epoch 2/128                                                                          

240/240 - 1s - 3ms/step - ia: 0.7206 - loss: 0.3044 - mae: 0.4048 - rmse: 0.5471 - smape: 0.7972 - val_ia: 0.7739 - val_loss: 0.1360 - val_mae: 0.2335 - val_rmse: 0.3341 - val_smape: 0.5457

Epoch 3/128                                                                          

240/240 - 1s - 3ms/step - ia: 0.7523 - loss: 0.2462 - mae: 0.3615 - rmse: 0.4920 - smape: 0.7337 - val_ia: 0.8005 - val_loss: 0.1136 - val_mae: 0.2076 - val_rmse: 0.3056 - val_smape: 0.4943

Epoch 4/128                                                                          

240/240 - 1s - 6ms/step - ia: 0.7718 - loss: 0.2126 - mae: 0.3347 - rmse: 0.45

In [23]:
print(best)

{'activation': 1, 'batch': 4, 'dropout': 0.2, 'layers': 1.0, 'learning_rate': 0.0002707756079796208, 'units': 4}
